In [ ]:
# !pip -q install -U --no-cache-dir transformers accelerate bitsandbytes
# !pip -q install --no-cache-dir --force-reinstall "Pillow==10.2.0"

In [ ]:
!pip -q install retina-face insightface onnxruntime-gpu

In [ ]:
!pip -q install -U open_clip_torch

In [ ]:
import os

# Traverse the directory tree starting at '.' (current directory)
for root, dirs, files in os.walk("/kaggle/input/"):
    # print(f"Root : {root}, dir: {dirs},total files:{len(files)}" )
    for name in sorted(files)[:2]:
        
        print(os.path.join(root, name))

In [ ]:
import os
import re
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import pandas as pd

In [ ]:
ROOT = "/kaggle/input/shared-poli-dataset/dataset"
TAMIL_IMG_DIR = os.path.join(ROOT, "Train_images(Tamil)")
MAL_IMG_DIR   = os.path.join(ROOT, "Train_images(Malayam)")  # note: 'Malayam' in folder name

TAMIL_XLSX = os.path.join(ROOT, "Tamil_Train_labels.xlsx")
MAL_XLSX   = os.path.join(ROOT, "Malayalam_Train_label.xlsx")

In [ ]:
# -------------------------
# 1) Load labels
# -------------------------
t = pd.read_excel(TAMIL_XLSX)
m = pd.read_excel(MAL_XLSX)

# Standardize columns
t = t.rename(columns={"Level1":"Level 1", "Level2":"Level 2"})
t["lang"] = "tamil"
m["lang"] = "malayalam"

# Tamil has Image_name; Malayalam has meme_id
# We'll create an image file name for both.
def tamil_candidates(row):
    # Most robust: try Image_name then also try numeric id as jpg
    cands = []
    if "Image_name" in row and pd.notna(row["Image_name"]):
        cands.append(str(row["Image_name"]).strip())
    if "Image_id" in row and pd.notna(row["Image_id"]):
        cands.append(f"{int(row['Image_id'])}.jpg")
    # Also try 3-digit zero-pad from Image_id
    if "Image_id" in row and pd.notna(row["Image_id"]):
        cands.append(f"{int(row['Image_id']):03d}.jpg")
    return list(dict.fromkeys(cands))

def mal_candidates(row):
    # Many images in folder appear as 208.jpg etc.
    # meme_id seems numeric -> "{id}.jpg"
    cands = []
    if "meme_id" in row and pd.notna(row["meme_id"]):
        cands.append(f"{int(row['meme_id'])}.jpg")
        # some datasets may also use zero-pad, so try just in case
        cands.append(f"{int(row['meme_id']):03d}.jpg")
    return list(dict.fromkeys(cands))

def resolve_path(img_dir, candidates):
    for fn in candidates:
        p = os.path.join(img_dir, fn)
        if os.path.exists(p):
            return p
    return None

# Build resolved image paths
t["image_path"] = t.apply(lambda r: resolve_path(TAMIL_IMG_DIR, tamil_candidates(r)), axis=1)
m["image_path"] = m.apply(lambda r: resolve_path(MAL_IMG_DIR,   mal_candidates(r)), axis=1)

In [ ]:
# -------------------------
# 2) Map to INTERNAL labels
# -------------------------
def map_internal(lang, level1, level2):
    # Level1 internal
    if lang == "tamil":
        stance = "SUPPORT" if level1 == "Support/Praise" else "TROLL"
        # Level2 internal
        if level2 == "Support for person" or level2 == "Troll/Oppose Against Person":
            target = "PERSON"
        elif level2 == "Support for party" or level2 == "Troll/Oppose Against Party":
            target = "PARTY"
        else:
            target = "UNKNOWN"
    else:
        stance = "SUPPORT" if str(level1).strip().upper() == "SUPPORT" else "TROLL"
        l2 = str(level2).strip()
        if l2 in ["Against individual person", "Support for individual person"]:
            target = "PERSON"
        elif l2 in ["Against party", "Support for party"]:
            target = "PARTY"
        elif l2 == "Intersection":
            target = "INTERSECTION"
        else:
            target = "UNKNOWN"
    return stance, target

def add_internal(df):
    stances, targets = [], []
    for _, r in df.iterrows():
        stance, target = map_internal(r["lang"], r["Level 1"], r["Level 2"])
        stances.append(stance); targets.append(target)
    df["stance_internal"] = stances
    df["target_internal"] = targets
    return df

t = add_internal(t)
m = add_internal(m)

# Keep only key columns
t_out = t[["lang", "image_path", "Level 1", "Level 2", "stance_internal", "target_internal"]].copy()
m_out = m[["lang", "image_path", "Level 1", "Level 2", "stance_internal", "target_internal"]].copy()

df = pd.concat([t_out, m_out], ignore_index=True)

# Quick sanity checks
missing = df["image_path"].isna().sum()
print("Total rows:", len(df))
print("Missing image_path:", missing)
if missing:
    print(df[df["image_path"].isna()].head(10))

In [ ]:
# -------------------------
# 3) Face detection (fast baseline)
# -------------------------
# Using OpenCV Haar cascade (fast and lightweight).
# Not best, but good enough for "face count" signal on Kaggle.
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def face_stats(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return 0, 0.0
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(40, 40)
    )
    num = len(faces)
    if num == 0:
        return 0, 0.0
    # area of largest face box relative to image area (helps robustness)
    h, w = gray.shape
    areas = [(fw * fh) / float(w * h) for (x, y, fw, fh) in faces]
    return num, float(max(areas))

num_faces_list = []
max_face_area_list = []

for p in tqdm(df["image_path"].fillna("").tolist()):
    if not p:
        num_faces_list.append(0)
        max_face_area_list.append(0.0)
        continue
    n, a = face_stats(p)
    num_faces_list.append(n)
    max_face_area_list.append(a)

df["num_faces"] = num_faces_list
df["max_face_area"] = max_face_area_list

In [ ]:
out_path = "/kaggle/working/train_with_faces.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

# Show useful summaries
print("\nFace count summary by language:")
print(df.groupby("lang")["num_faces"].describe())

print("\nMalayalam Intersection vs num_faces:")
mal = df[df["lang"]=="malayalam"]
print(pd.crosstab(mal["target_internal"], mal["num_faces"]>=2, rownames=["target_internal"], colnames=[">=2 faces"]))

In [ ]:
faces = pd.read_csv("/kaggle/working/train_with_faces.csv")
faces

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("/kaggle/working/train_with_faces.csv")

# simple feature set for now
X = df[["num_faces", "max_face_area"]].copy()

def eval_task(y, name):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000))
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X, y, cv=cv, scoring="f1_macro")
    print(f"{name} macro-F1: {scores.mean():.4f} ± {scores.std():.4f}")

eval_task(df["stance_internal"], "STANCE (SUPPORT vs TROLL)")
eval_task(df["target_internal"], "TARGET (PERSON/PARTY/INTERSECTION)")

In [ ]:
!pip -q install -U open_clip_torch

In [79]:
import torch
import open_clip
from PIL import Image
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

model_clip, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14",
    pretrained="laion2b_s32b_b82k"
)

model_clip = model_clip.to(device).eval()

In [80]:
def embed_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img_t = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = model_clip.encode_image(img_t)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze(0).cpu().numpy()

In [81]:
from tqdm import tqdm

img_embs = []

for p in tqdm(df["image_path"].tolist(), desc="Embedding train memes"):
    img_embs.append(embed_image(p))

img_embs = np.vstack(img_embs)

print("Train image embedding shape:", img_embs.shape)

Embedding train memes: 100%|██████████| 1303/1303 [01:38<00:00, 13.20it/s]

Train image embedding shape: (1303, 768)


In [96]:
# STEP 2 — Load and embed ALL logos (open_clip version)

import os
import numpy as np
import torch
from PIL import Image

LOGO_ROOT = "/kaggle/input/shared-task-political-party-logos/logos"

logo_embs = []
logo_party = []

skipped = 0
loaded = 0

for party in sorted(os.listdir(LOGO_ROOT)):
    party_dir = os.path.join(LOGO_ROOT, party)
    if not os.path.isdir(party_dir):
        continue
        
    for f in os.listdir(party_dir):
        # ❌ skip SVGs (PIL can't load them)
        if not f.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".avif")):
            continue
        
        img_path = os.path.join(party_dir, f)
        try:
            img = Image.open(img_path).convert("RGB")

            img_t = preprocess(img).unsqueeze(0).to(device)

            with torch.no_grad():
                emb = model_clip.encode_image(img_t)
                emb = emb / emb.norm(dim=-1, keepdim=True)

            logo_embs.append(emb.squeeze(0).cpu().numpy())
            logo_party.append(party)
            loaded += 1

        except Exception as e:
            skipped += 1
            # Uncomment next line if you want to debug first failure
            # print("Failed:", img_path, type(e).__name__, e)

print(f"Loaded logos: {loaded}")
print(f"Skipped logos: {skipped}")

# Safety check
assert loaded > 0, "No logo embeddings were created — check paths / formats"

logo_embs = np.vstack(logo_embs)
logo_party = np.array(logo_party)

print("Final logo embedding shape:", logo_embs.shape)

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Loaded logos: 30
Skipped logos: 0
Final logo embedding shape: (30, 768)


In [174]:
# STEP 3 — Compute logo similarity for EACH meme image

# We turn logo similarity into 4 numeric features.

# Features we extract:

# logo_max_sim

# logo_top2_gap

# logo_hits (count above threshold)

# logo_party_id (encoded index)

from sklearn.preprocessing import LabelEncoder

party_encoder = LabelEncoder()
party_encoder.fit(logo_party)

def logo_features(img_emb, thresh=0.25):
    sims = img_emb @ logo_embs.T
    top1 = sims.max()
    top2 = np.partition(sims, -2)[-2]
    return [
        float(top1),
        float(top1 - top2),
        int((sims > thresh).sum())
    ]


In [114]:
# rebuild logo features (now only 3 cols)
logo_feats = np.array(
    [logo_features(e) for e in img_embs],
    dtype=np.float32
)

face_feats = df[["num_faces", "max_face_area"]].to_numpy().astype(np.float32)

X_logo = np.hstack([img_embs, face_feats, logo_feats]).astype(np.float32)

print("X_logo shape:", X_logo.shape)


X_logo shape: (1303, 774)


In [115]:
# STEP 4 — Add logo features to TRAIN set
logo_feat_list = []

for emb in img_embs:
    logo_feat_list.append(logo_features(emb))

logo_feats = np.array(logo_feat_list)

print("Logo feature shape:", logo_feats.shape)

Logo feature shape: (1303, 4)


In [116]:
print("Logo max sim stats:")
print(
    logo_feats[:,0].min(),
    logo_feats[:,0].mean(),
    logo_feats[:,0].max()
)

Logo max sim stats:
0.19605191051959991 0.3872649832267717 0.6393736600875854


In [117]:
# STEP 5 — Build FINAL training matrix
X = np.hstack([
    img_embs,
    df[["num_faces","max_face_area"]].values.astype(np.float32),
    logo_feats.astype(np.float32)
])

y_stance = df["stance_internal"].values
y_target = df["target_internal"].values

In [95]:
# STEP 6 — Train classifiers (same as before)
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
np.random.seed(42)
clf_stance = LinearSVC(class_weight="balanced" , max_iter=20000, dual = True)
clf_target = LinearSVC(class_weight="balanced" , max_iter=20000, dual = True)

clf_stance.fit(X, y_stance)
clf_target.fit(X, y_target)

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVC(class_weight='balanced', dual=True, max_iter=20000)

In [118]:
# STEP 6.5

# --- Build logo feature matrix for TRAIN ---

logo_feat_list = []
for emb in img_embs:
    logo_feat_list.append(logo_features(emb, thresh=0.25))

logo_feats = np.array(logo_feat_list, dtype=np.float32)   # (N, 4)

# --- Face features (use same ones you trained with before) ---
face_feats = df[["num_faces", "max_face_area"]].to_numpy().astype(np.float32)  # (N, 2)

# --- Final X: [CLIP | FACE | LOGO] ---
X_logo = np.hstack([img_embs.astype(np.float32), face_feats, logo_feats])

print("X_logo shape:", X_logo.shape)

X_logo shape: (1303, 774)


In [119]:
print("CLIP dim:", img_embs.shape[1])
print("FACE dim:", face_feats.shape[1])
print("LOGO dim:", logo_feats.shape[1])

CLIP dim: 768
FACE dim: 2
LOGO dim: 4


In [120]:
df = df.reset_index(drop=True).copy()

img_embs = img_embs.astype(np.float32)
face_feats = df[["num_faces","max_face_area"]].to_numpy().astype(np.float32)
logo_feats = logo_feats.astype(np.float32)

X_logo = np.hstack([img_embs, face_feats, logo_feats]).astype(np.float32)

print("X_logo md5:", hash_array(X_logo))


X_logo md5: 001939cd853522de4e09091919814d82


In [121]:
#Step 6.6
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC

def cv_macro_f1(X, y):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    model = LinearSVC(class_weight="balanced")
    pred = cross_val_predict(model, X, y, cv=skf)
    return float(f1_score(y, pred, average="macro"))

stance_f1 = cv_macro_f1(X_logo, df["stance_internal"].values)
target_f1 = cv_macro_f1(X_logo, df["target_internal"].values)

print("OVERALL STANCE macro-F1 (CLIP+FACE+LOGO):", stance_f1)
print("OVERALL TARGET macro-F1 (CLIP+FACE+LOGO):", target_f1)

OVERALL STANCE macro-F1 (CLIP+FACE+LOGO): 0.7754534182376058
OVERALL TARGET macro-F1 (CLIP+FACE+LOGO): 0.47243797844208957


In [122]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import f1_score

def cv_macro_f1_lang(lang, col):
    mask = (df["lang"].values == lang)
    Xsub = X_logo[mask]
    ysub = df.loc[mask, col].values

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", LinearSVC(
            class_weight="balanced",
            max_iter=20000   
        ))
    ])

    out = cross_validate(
        model,
        Xsub,
        ysub,
        cv=skf,
        scoring="f1_macro",
        n_jobs=1
    )

    return out["test_score"].mean(), out["test_score"].std()

for lang in ["tamil", "malayalam"]:
    s_mean, s_std = cv_macro_f1_lang(lang, "stance_internal")
    t_mean, t_std = cv_macro_f1_lang(lang, "target_internal")

    print(lang)
    print("STANCE macro-F1:", s_mean, "±", s_std)
    print("TARGET macro-F1:", t_mean, "±", t_std)
    print()


tamil
STANCE macro-F1: 0.7630067421659394 ± 0.02309924589206219
TARGET macro-F1: 0.5839870947055067 ± 0.04262999667392781

malayalam
STANCE macro-F1: 0.6304208836826418 ± 0.03425852570597073
TARGET macro-F1: 0.45322344160768474 ± 0.02391759791489555



In [123]:
print("df shape:", df.shape)
print("X_logo shape:", X_logo.shape)

# Check if rows still align
print("first 5 image paths:")
print(df["image_path"].head().tolist())

# Check if X_logo has NaN/inf
print("X_logo finite:", np.isfinite(X_logo).all())

df shape: (1303, 8)
X_logo shape: (1303, 774)
first 5 image paths:
['/kaggle/input/shared-poli-dataset/dataset/Train_images(Tamil)/000.jpg', '/kaggle/input/shared-poli-dataset/dataset/Train_images(Tamil)/002.jpg', '/kaggle/input/shared-poli-dataset/dataset/Train_images(Tamil)/003.jpg', '/kaggle/input/shared-poli-dataset/dataset/Train_images(Tamil)/004.jpg', '/kaggle/input/shared-poli-dataset/dataset/Train_images(Tamil)/005.jpg']
X_logo finite: True


In [124]:
import numpy as np, hashlib

def hash_array(a):
    a = np.ascontiguousarray(a)
    return hashlib.md5(a.tobytes()).hexdigest()

print("X_logo md5:", hash_array(X_logo))

X_logo md5: 001939cd853522de4e09091919814d82


In [125]:
# STEP 7 — Predict TEST images (IMPORTANT)

# You must extract logo features for test images the same way.

@torch.inference_mode()
def extract_test_features(img_path):
    img = Image.open(img_path).convert("RGB")
    img_t = preprocess(img).unsqueeze(0).to(device)

    emb = model_clip.encode_image(img_t)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    emb = emb.squeeze(0).cpu().numpy()

    lf = logo_features(emb, thresh=0.25)
    n, a = face_stats_cv2(img_path)

    return np.hstack([emb, [n, a], lf]).astype(np.float32)


In [126]:
def face_stats_cv2(path):
    img = cv2.imread(path)
    if img is None:
        return 0, 0.0
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    num = len(faces)
    if num == 0:
        return 0, 0.0
    h, w = gray.shape
    areas = [(fw * fh) / (w * h) for (x, y, fw, fh) in faces]
    return num, float(max(areas))

In [152]:
# ============================================================================
# Train final models (per language, LinearSVC)
# ============================================================================
print("\n>>> PHASE 7: Training final models (per language)...")

def train_models_for_lang(lang):
    mask = (df["lang"].values == lang)
    Xsub = X[mask]
    y_stance = df.loc[mask, "stance_internal"].values
    y_target = df.loc[mask, "target_internal"].values

    m_stance = LinearSVC(class_weight="balanced", max_iter=50000)
    m_target = LinearSVC(class_weight="balanced", max_iter=50000)

    m_stance.fit(Xsub, y_stance)
    m_target.fit(Xsub, y_target)
    
    print(f"  {lang.upper()} models trained")
    return m_stance, m_target

m_ta_stance, m_ta_target = train_models_for_lang("tamil")
m_ml_stance, m_ml_target = train_models_for_lang("malayalam")

print("✅ All models trained")


>>> PHASE 7: Training final models (per language)...
  TAMIL models trained


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  MALAYALAM models trained
✅ All models trained


In [153]:
import numpy as np
from PIL import Image
import torch

@torch.inference_mode()
def embed_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img_t = preprocess(img).unsqueeze(0).to(device)
    emb = model_clip.encode_image(img_t)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze(0).cpu().numpy()

def extract_one_features(img_path, thresh=0.25):
    emb = embed_image(img_path)                 # (D,)
    lf  = np.array(logo_features(emb, thresh=thresh), dtype=np.float32)  # (4,)
    nf, mfa = face_stats_cv2(img_path)          # (2,)
    face = np.array([nf, mfa], dtype=np.float32)
    return np.hstack([emb, face, lf]).astype(np.float32)

In [154]:
# -----------------------------
# 3) Paths (train/test)
# -----------------------------
TRAIN_WITH_FACES_CSV = "/kaggle/working/train_with_faces.csv"

TAMIL_TEST_IMG_DIR = "/kaggle/input/shared-task-meme-test/test/Tamil_test/Tamil_Test_images"
MAL_TEST_IMG_DIR   = "/kaggle/input/shared-task-meme-test/test/Malaylam_test/Malaylam_Test_images"

TAMIL_TEST_CSV = "/kaggle/input/shared-task-meme-test/test/Tamil_test/Tamil_test_labels.csv"
MAL_TEST_CSV   = "/kaggle/input/shared-task-meme-test/test/Malaylam_test/Malayalam_Test_labels.csv"

In [155]:
import os
import pandas as pd
from tqdm import tqdm



# load test CSVs
tam = pd.read_csv(TAMIL_TEST_CSV)
mal = pd.read_csv(MAL_TEST_CSV)

tam_paths = [os.path.join(TAMIL_TEST_IMG_DIR, str(n).strip()) for n in tam["Image_name"]]
mal_paths = [os.path.join(MAL_TEST_IMG_DIR, f"{int(i)}.jpg") for i in mal["meme_id"]]

def build_X(paths):
    feats = []
    for p in tqdm(paths, desc="Building test features"):
        feats.append(extract_one_features(p))
    return np.vstack(feats)

X_tam_test = build_X(tam_paths)
X_mal_test = build_X(mal_paths)

print("Tamil test X:", X_tam_test.shape)
print("Malayalam test X:", X_mal_test.shape) 

Building test features: 100%|██████████| 100/100 [00:35<00:00,  2.85it/s]

Tamil test X: (201, 774)
Malayalam test X: (100, 774)


In [157]:
# ============================================================================
# CELL 11: Define mapping functions for output
# ============================================================================
print("\n>>> PHASE 8: Defining output mappings...")

def map_to_tamil(stance, target):
    if stance == "SUPPORT":
        lvl1 = "Support/Praise"
        if target == "PARTY":
            lvl2 = "Support for party"
        else:
            lvl2 = "Support for person"
    else:
        lvl1 = "Troll/Oppose"
        if target == "PARTY":
            lvl2 = "Troll/Oppose Against Party"
        else:
            lvl2 = "Troll/Oppose Against Person"
    return lvl1, lvl2

def map_to_malayalam(stance, target):
    if stance == "SUPPORT":
        lvl1 = "SUPPORT"
        if target == "PERSON":
            lvl2 = "Support for individual person"
        elif target == "PARTY":
            lvl2 = "Support for party"
        else:
            lvl2 = "Intersection"
    else:
        lvl1 = "TROLL/ OPPOSE"
        if target == "PERSON":
            lvl2 = "Against individual person"
        elif target == "PARTY":
            lvl2 = "Against party"
        else:
            lvl2 = "Intersection"
    return lvl1, lvl2

print("✅ Mapping functions defined")


>>> PHASE 8: Defining output mappings...
✅ Mapping functions defined


In [158]:
pred_ta_stance = m_ta_stance.predict(X_tam_test)
pred_ta_target = m_ta_target.predict(X_tam_test)

pred_ml_stance = m_ml_stance.predict(X_mal_test)
pred_ml_target = m_ml_target.predict(X_mal_test)

In [159]:
lvl1, lvl2 = [], []
for s, t in zip(pred_ta_stance, pred_ta_target):
    a, b = map_to_tamil(s, t)
    lvl1.append(a); lvl2.append(b)

tam["Level1"] = lvl1
tam["Level2"] = lvl2

out_tam = "/kaggle/working/Tamil_test_predictions_CLIP_SVC_LOGO.csv"
tam.to_csv(out_tam, index=False)
print("Saved:", out_tam)

Saved: /kaggle/working/Tamil_test_predictions_CLIP_SVC_LOGO.csv


In [160]:
lvl1, lvl2 = [], []
for s, t in zip(pred_ml_stance, pred_ml_target):
    a, b = map_to_malayalam(s, t)
    lvl1.append(a); lvl2.append(b)

mal["Level 1"] = lvl1
mal["Level 2"] = lvl2

out_mal = "/kaggle/working/Malayalam_test_predictions_CLIP_SVC_LOGO.csv"
mal.to_csv(out_mal, index=False)
print("Saved:", out_mal)

Saved: /kaggle/working/Malayalam_test_predictions_CLIP_SVC_LOGO.csv


In [179]:
!zip -r "with logo.zip" "/kaggle/working/"

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 73%)
  adding: kaggle/working/Tamil_test_predictions_CLIP_SVC_LOGO.csv (deflated 88%)
  adding: kaggle/working/train_with_faces.csv (deflated 91%)
  adding: kaggle/working/Malayalam_test_predictions_CLIP_SVC_LOGO.csv (deflated 88%)


In [161]:
print("CLIP dim:", embed_image(df["image_path"].iloc[0]).shape)
print("logo_embs shape:", logo_embs.shape)

CLIP dim: (768,)
logo_embs shape: (30, 768)


In [168]:
import numpy as np

# 1) Face features (same as before)
face_feats = df[["num_faces", "max_face_area"]].to_numpy().astype(np.float32)

# 2) X_base = CLIP + FACE
X_base = np.hstack([img_embs, face_feats]).astype(np.float32)

# 3) logo_feats must already be computed using the 3-value logo_features()
# If you have logo_feats already, keep it. Otherwise uncomment next lines:

# logo_feats = np.array([logo_features(e) for e in img_embs], dtype=np.float32)

# 4) X_logo = CLIP + FACE + LOGO
X_logo = np.hstack([img_embs, face_feats, logo_feats]).astype(np.float32)



X_base shape: (1303, 770)
X_logo shape: (1303, 774)


In [175]:
logo_feats = np.array([logo_features(e) for e in img_embs], dtype=np.float32)  # should be (N, 3)
X_logo = np.hstack([img_embs, face_feats, logo_feats]).astype(np.float32)

print("logo_feats shape:", logo_feats.shape)
print("X_logo shape:", X_logo.shape)  # should be (N, 773)


logo_feats shape: (1303, 3)
X_logo shape: (1303, 773)


In [176]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

def cv_target_one_stage(X, y):
    y = np.asarray(y)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pred_all = np.empty(len(y), dtype=object)

    for tr, te in skf.split(X, y):
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", LinearSVC(class_weight="balanced", max_iter=20000))
        ])
        model.fit(X[tr], y[tr])
        pred_all[te] = model.predict(X[te])

    return float(f1_score(y, pred_all, average="macro"))

def cv_two_stage_target_safe(X, y_target):
    y_target = np.asarray(y_target)
    y_inter = (y_target == "INTERSECTION").astype(int)

    # If there is no INTERSECTION at all, two-stage is impossible -> fallback
    if np.unique(y_inter).size < 2:
        return cv_target_one_stage(X, y_target)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pred_all = np.empty(len(y_target), dtype=object)

    for tr, te in skf.split(X, y_inter):
        X_tr, X_te = X[tr], X[te]
        y_tr_target = y_target[tr]
        y_tr_inter  = y_inter[tr]

        # ---- Stage 1: INTERSECTION vs NOT ----
        clf_inter = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", LinearSVC(class_weight="balanced", max_iter=20000))
        ])
        clf_inter.fit(X_tr, y_tr_inter)
        pred_inter = clf_inter.predict(X_te).astype(bool)

        te = np.asarray(te)
        pred_all[te[pred_inter]] = "INTERSECTION"

        idx_not = te[~pred_inter]
        if len(idx_not) > 0:
            # ---- Stage 2: PERSON vs PARTY (train on non-intersection only) ----
            mask_pp = (y_tr_target != "INTERSECTION")

            # If stage-2 would have only one class, fallback to most common
            y_pp = y_tr_target[mask_pp]
            X_pp = X_tr[mask_pp]
            if len(np.unique(y_pp)) < 2:
                pred_all[idx_not] = y_pp[0]
            else:
                clf_pp = Pipeline([
                    ("scaler", StandardScaler()),
                    ("svc", LinearSVC(class_weight="balanced", max_iter=20000))
                ])
                clf_pp.fit(X_pp, y_pp)
                pred_all[idx_not] = clf_pp.predict(X[idx_not])

    return float(f1_score(y_target, pred_all, average="macro"))


In [178]:
for lang in ["tamil", "malayalam"]:
    mask = (df["lang"].values == lang)
    y = df.loc[mask, "target_internal"].values

    f1_base = cv_two_stage_target_safe(X_base[mask], y)
    f1_logo = cv_two_stage_target_safe(X_logo[mask], y)

    print("\n", lang.upper())
    print("Two-stage TARGET F1 (X_base):", f1_base)
    print("Two-stage TARGET F1 (X_logo):", f1_logo)



 TAMIL
Two-stage TARGET F1 (X_base): 0.5861985740657392
Two-stage TARGET F1 (X_logo): 0.5827065789101757

 MALAYALAM
Two-stage TARGET F1 (X_base): 0.4658860265417643
Two-stage TARGET F1 (X_logo): 0.4694310803545581


In [ ]:
import os

# Traverse the directory tree starting at '.' (current directory)
for root, dirs, files in os.walk("/kaggle/input"):
    # print(f"Root : {root}, dir: {dirs},total files:{len(files)}" )
    for name in sorted(files)[:3]:
        
        print(os.path.join(root, name))

In [ ]:
import os

# Traverse the directory tree starting at '.' (current directory)
for root, dirs, files in os.walk("/kaggle/input/shared-task-meme-test"):
    # print(f"Root : {root}, dir: {dirs},total files:{len(files)}" )
    for name in sorted(files)[:3]:
        
        print(os.path.join(root, name))

In [ ]:
import os

# Traverse the directory tree starting at '.' (current directory)
for root, dirs, files in os.walk("/kaggle/input/shared-task-political-party-logos"):
    # print(f"Root : {root}, dir: {dirs},total files:{len(files)}" )
    for name in sorted(files)[:3]:
        
        print(os.path.join(root, name))